<a href="https://colab.research.google.com/github/amaimanwar8-arch/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/amaimanwar8-arch/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print("Working dir:", os.getcwd())
print(df.shape[0], "pages loaded successfully.")

Working dir: /content/flyrank-ml-internship
30000 pages loaded successfully.


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [2]:
visible = df[df["impressions_90d"] >= 100].copy()

# Signal 1 (flag-linked: behind CTR-fix / low_ctr_visible_page logic)
sig1 = visible.groupby("position_tier")["ctr"].agg(["mean", "count"]).rename(columns={"mean": "avg_ctr", "count": "n"})
print("Signal 1 - CTR by position tier (assumption behind CTR-fix logic: CTR must be compared WITHIN a tier, not globally):")
print(sig1)

# Signal 2 (flag-linked: behind refresh flags / stale_visible_page)
sig2 = df.groupby("freshness_tier")["is_declining_label"].agg(["mean", "count"]).rename(columns={"mean": "decline_rate", "count": "n"})
print("\nSignal 2 - decline rate by freshness tier (assumption behind stale_visible_page: staler pages decline more):")
print(sig2)

Signal 1 - CTR by position tier (assumption behind CTR-fix logic: CTR must be compared WITHIN a tier, not globally):
                avg_ctr     n
position_tier                
deep           0.055415   879
page_1         0.354760  8633
page_3_5       0.142359  6058
striking       0.255782  5903
top_3          0.334128   533

Signal 2 - decline rate by freshness tier (assumption behind stale_visible_page: staler pages decline more):
                decline_rate      n
freshness_tier                     
0-30                0.511377  20480
181+                0.471264    174
31-90               0.588571    175
91-180              0.611057   9171


In [3]:
print("""
Signal 1 - CTR by position tier: CONFIRMED.
CTR drops sharply and monotonically from page_1 (0.355) down to deep (0.055), with solid
sample sizes at every tier (533-8,633 rows). This confirms the CTR-fix flag's core
assumption: CTR must be judged relative to its own position tier, never compared globally,
since a CTR that looks weak at top_3 would look excellent at deep.

Signal 2 - decline rate by freshness tier: MIXED.
Decline rate rises from 0-30 days (0.511) through 91-180 days (0.611) as expected - but
then reverses at 181+ days (0.471), the LOWEST of all four tiers. The simple "staler pages
decline more" assumption behind stale_visible_page does NOT hold cleanly across the full
range. Worth noting: the 31-90 (n=175) and 181+ (n=174) buckets are far smaller than 0-30
(n=20,480), so the reversal at the tail deserves caution rather than full trust either way.
Because of this, my rule will lean on the 91-180 peak rather than "staleness always wins."

My rule: score pages higher when they combine (a) weak CTR relative to their OWN position
tier, since Signal 1 held cleanly, and (b) freshness_tier == '91-180' specifically (the
confirmed peak-decline zone), rather than a blanket "old = bad" assumption Signal 2 just
disproved.

Reason code: ctr_gap_and_aging_review
Action label: review_for_refresh
""")


Signal 1 - CTR by position tier: CONFIRMED.
CTR drops sharply and monotonically from page_1 (0.355) down to deep (0.055), with solid
sample sizes at every tier (533-8,633 rows). This confirms the CTR-fix flag's core
assumption: CTR must be judged relative to its own position tier, never compared globally,
since a CTR that looks weak at top_3 would look excellent at deep.

Signal 2 - decline rate by freshness tier: MIXED.
Decline rate rises from 0-30 days (0.511) through 91-180 days (0.611) as expected - but
then reverses at 181+ days (0.471), the LOWEST of all four tiers. The simple "staler pages
decline more" assumption behind stale_visible_page does NOT hold cleanly across the full
range. Worth noting: the 31-90 (n=175) and 181+ (n=174) buckets are far smaller than 0-30
(n=20,480), so the reversal at the tail deserves caution rather than full trust either way.
Because of this, my rule will lean on the 91-180 peak rather than "staleness always wins."

My rule: score pages higher when

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [4]:
scored = df[df["impressions_90d"] >= 100].copy()

# Expected CTR per tier, from Signal 1's real numbers above
tier_avg_ctr = scored.groupby("position_tier")["ctr"].transform("mean")
scored["ctr_gap"] = (tier_avg_ctr - scored["ctr"]).clip(lower=0)  # how far BELOW its own tier's average

# Aging signal: use the confirmed peak zone from Signal 2, not a blanket "old=bad"
scored["aging_flag"] = (scored["freshness_tier"] == "91-180").astype(int)

# Normalize ctr_gap to 0-1 so it combines fairly with the binary aging_flag
scored["ctr_gap_norm"] = scored["ctr_gap"] / scored["ctr_gap"].max()

scored["baseline_action_score"] = (0.65 * scored["ctr_gap_norm"]) + (0.35 * scored["aging_flag"])

scored["reason_code"] = "ctr_gap_and_aging_review"
scored["action"] = "review_for_refresh"

queue = scored.sort_values("baseline_action_score", ascending=False)[
    ["content_id", "position_tier", "ctr", "ctr_gap", "freshness_tier",
     "baseline_action_score", "reason_code", "action"]
]

os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)

print(f"Wrote {len(queue):,} ranked rows to work/outputs/baseline_action_score.csv")
queue.head(10)

Wrote 22,006 ranked rows to work/outputs/baseline_action_score.csv


,content_id,position_tier,ctr,ctr_gap,freshness_tier,baseline_action_score,reason_code,action
21358,content_5352bbcea612,page_1,0.0,0.35476,91-180,1.0,ctr_gap_and_aging_review,review_for_refresh
9646,content_384b06d9d970,page_1,0.0,0.35476,91-180,1.0,ctr_gap_and_aging_review,review_for_refresh
22374,content_761694ab97b4,page_1,0.0,0.35476,91-180,1.0,ctr_gap_and_aging_review,review_for_refresh
4156,content_6ff59596237d,page_1,0.0,0.35476,91-180,1.0,ctr_gap_and_aging_review,review_for_refresh
24472,content_8db4c9e4360b,page_1,0.0,0.35476,91-180,1.0,ctr_gap_and_aging_review,review_for_refresh
24473,content_fa53b1771cb3,page_1,0.0,0.35476,91-180,1.0,ctr_gap_and_aging_review,review_for_refresh
19253,content_78b6f55b4f70,page_1,0.0,0.35476,91-180,1.0,ctr_gap_and_aging_review,review_for_refresh
24483,content_82b0db79f37e,page_1,0.0,0.35476,91-180,1.0,ctr_gap_and_aging_review,review_for_refresh
15628,content_e8783e661965,page_1,0.0,0.35476,91-180,1.0,ctr_gap_and_aging_review,review_for_refresh
120,content_8f2559c3bc1b,page_1,0.0,0.35476,91-180,1.0,ctr_gap_and_aging_review,review_for_refresh


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [5]:
top10 = queue.head(10).reset_index(drop=True)

for i, row in top10.iterrows():
    print(f"{i+1}. {row['content_id']} | action: {row['action']} | reason: {row['reason_code']}")
    print(f"   Why: ranks {row['position_tier']}, but CTR={row['ctr']:.4f} vs. its tier's average "
          f"(gap={row['ctr_gap']:.4f}); also sits in the 91-180 day aging zone.")
    print(f"   What would make this wrong: if this page's true audience genuinely converts off-SERP "
          f"(e.g. brand navigational query where users don't need to click through), a 0.0 CTR isn't")
    print(f"   a content problem at all - it would need a query-intent check before acting on it.\n")

1. content_5352bbcea612 | action: review_for_refresh | reason: ctr_gap_and_aging_review
   Why: ranks page_1, but CTR=0.0000 vs. its tier's average (gap=0.3548); also sits in the 91-180 day aging zone.
   What would make this wrong: if this page's true audience genuinely converts off-SERP (e.g. brand navigational query where users don't need to click through), a 0.0 CTR isn't
   a content problem at all - it would need a query-intent check before acting on it.

2. content_384b06d9d970 | action: review_for_refresh | reason: ctr_gap_and_aging_review
   Why: ranks page_1, but CTR=0.0000 vs. its tier's average (gap=0.3548); also sits in the 91-180 day aging zone.
   What would make this wrong: if this page's true audience genuinely converts off-SERP (e.g. brand navigational query where users don't need to click through), a 0.0 CTR isn't
   a content problem at all - it would need a query-intent check before acting on it.

3. content_761694ab97b4 | action: review_for_refresh | reason: ctr_g

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [6]:
tied_count = (queue["baseline_action_score"] == queue["baseline_action_score"].max()).sum()

print(f"""
Weak picks: my top-10 all tied at the maximum possible score (1.0), and {tied_count:,} rows
share this same tied maximum across the full dataset. This isn't 10 meaningfully different
"worst-of-the-worst" pages - it's one large tied cluster of page_1 pages with exactly
ctr=0.0000 in the 91-180 aging zone. My scoring rule can't currently distinguish WITHIN
that tied group, since ctr_gap_norm caps at 1.0 for anyone with zero clicks, and aging_flag
is binary. A real reviewer using this queue would need a tie-breaker (e.g. impressions_90d,
so the tied page getting the MOST exposure with zero clicks gets reviewed first) before
this ranking is genuinely actionable at the top.

Leakage check: my two inputs, ctr_gap (built from ctr and position_tier) and aging_flag
(built from freshness_tier), are both OBSERVED signals available before any decision point
- neither is a future-window metric and neither is trend_direction/trend_pct/is_declining_label,
which I've confirmed in earlier notebooks are label-derived and excluded from feature use.
No product flags (health_score, priority_score, action_type) exist in this dataset to leak
in the first place. The rule is leakage-safe.
""")


Weak picks: my top-10 all tied at the maximum possible score (1.0), and 425 rows
share this same tied maximum across the full dataset. This isn't 10 meaningfully different
"worst-of-the-worst" pages - it's one large tied cluster of page_1 pages with exactly
ctr=0.0000 in the 91-180 aging zone. My scoring rule can't currently distinguish WITHIN
that tied group, since ctr_gap_norm caps at 1.0 for anyone with zero clicks, and aging_flag
is binary. A real reviewer using this queue would need a tie-breaker (e.g. impressions_90d,
so the tied page getting the MOST exposure with zero clicks gets reviewed first) before
this ranking is genuinely actionable at the top.

Leakage check: my two inputs, ctr_gap (built from ctr and position_tier) and aging_flag
(built from freshness_tier), are both OBSERVED signals available before any decision point
- neither is a future-window metric and neither is trend_direction/trend_pct/is_declining_label,
which I've confirmed in earlier notebooks are label-der

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.